# Project showcase: render on a free Kaggle GPU

Runs the `project-showcase` render pipeline (IFC to Blender Cycles frames) on Kaggle's free GPUs, about 30 GPU hours a week at no cost.

**Before you run:** in the right-hand panel set **Accelerator: GPU T4 x2** and turn **Internet on** (Kaggle asks for a phone-verified account for internet). Then *Run all*.

**Two modes** (set in the next cell):
- `PROJECT = "sample"`: downloads buildingSMART's public sample house and renders `bench.json`, a fixed benchmark. Use this first to measure the speed-up against a laptop.
- `PROJECT = "/kaggle/input/<your-dataset>"`: a Kaggle dataset holding a project folder (`shots.json`, `finishes.csv`, `work/model.obj`, `work/scene.json`, textures). **Client designs:** this uploads the model to Google's servers. Only do it with the developer's and architect's OK, and keep the dataset private.

Output: `/kaggle/working/frames.zip` (the `site/frames` folder, ready to drop into the site) plus timings. Download it from the Output tab.

In [ ]:
REPO = "https://github.com/TaigerToolbox/TaigerDesignClaude.git"
BRANCH = "claude/stoic-dirac-x6h619"        # change to main once merged
PROJECT = "sample"                           # or "/kaggle/input/<dataset>/<project folder>"
SHOTS = "bench.json"                         # shots file inside the project
DEVICE = "GPU"                               # "CPU" to compare on the same box
BLENDER = "5.2.3"                            # same version as the laptop, so results match

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: set Accelerator to GPU in the right panel"

In [ ]:
# Blender for Linux, plus the few system libraries headless Blender needs
import os, subprocess
BL_DIR = f"/tmp/blender-{BLENDER}-linux-x64"
if not os.path.exists(BL_DIR):
    major = ".".join(BLENDER.split(".")[:2])
    url = f"https://download.blender.org/release/Blender{major}/blender-{BLENDER}-linux-x64.tar.xz"
    subprocess.run(f"wget -q {url} -O /tmp/blender.tar.xz && tar -xf /tmp/blender.tar.xz -C /tmp && rm /tmp/blender.tar.xz", shell=True, check=True)
subprocess.run("apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libxkbcommon0 libsm6 libgl1 ffmpeg >/dev/null 2>&1", shell=True)
BL = f"{BL_DIR}/blender"
print(subprocess.run([BL, "--version"], capture_output=True, text=True).stdout.splitlines()[0])
print("ffmpeg:", subprocess.run("which ffmpeg", shell=True, capture_output=True, text=True).stdout.strip() or "MISSING")

In [ ]:
# The pipeline scripts, straight from the (public) repo
import shutil
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO, "/kaggle/working/repo"], check=True)
SKILL = "/kaggle/working/repo/.claude/skills/project-showcase"
print(os.listdir(SKILL + "/scripts"))

In [ ]:
# Project folder: the sample house built from the public IFC, or a copy of your dataset (inputs are read-only)
PROJ = "/kaggle/working/project"
shutil.rmtree(PROJ, ignore_errors=True)
if PROJECT == "sample":
    shutil.copytree(SKILL + "/examples/sample-house", PROJ)
    ifc = "https://raw.githubusercontent.com/buildingSMART/Sample-Test-Files/main/IFC%204.0.2.1%20(IFC%204%20ADD2%20TC1)/Simple-Scene/Building-Architecture.ifc"
    subprocess.run(f"pip -q install ifcopenshell && wget -q '{ifc}' -O {PROJ}/model.ifc", shell=True, check=True)
    r = subprocess.run(["python", SKILL + "/scripts/ifc_to_obj.py", PROJ + "/model.ifc", PROJ + "/work"], capture_output=True, text=True)
    print(r.stdout[-800:], r.stderr[-800:])
else:
    shutil.copytree(PROJECT, PROJ)
if not os.path.exists(PROJ + "/site"):
    shutil.copytree(SKILL + "/template", PROJ + "/site")
print(sorted(os.listdir(PROJ)))

In [ ]:
# Render. The device is forced in a copy of the shots file, so the same file works on laptop and Kaggle.
import json, time
cfg = json.load(open(f"{PROJ}/{SHOTS}"))
cfg["device"] = DEVICE
run_cfg = f"{PROJ}/_kaggle_{SHOTS}"
json.dump(cfg, open(run_cfg, "w"), indent=1)
t0 = time.time()
p = subprocess.run([BL, "-b", "--factory-startup", "-P", SKILL + "/scripts/blender_render.py", "--", run_cfg],
                   capture_output=True, text=True)
elapsed = time.time() - t0
keep = [l for l in p.stdout.splitlines() if any(k in l for k in ("GPU render", "frames,", "manifest", "Error", "CAMERA", "UNMAPPED", "skipped", "sun:"))]
print(*keep, sep=chr(10)); print(p.stderr[-2000:] if p.returncode else "")
print(f"Exit {p.returncode}. Total {elapsed/60:.1f} min on {DEVICE} (includes scene build).")

In [ ]:
# Package the frames and show one
from IPython.display import Image as Show, display
shutil.make_archive("/kaggle/working/frames", "zip", PROJ + "/site", "frames")
print("frames.zip:", round(os.path.getsize("/kaggle/working/frames.zip") / 1e6, 1), "MB -> Output tab")
stills = [os.path.join(dp, f) for dp, _, fs in os.walk(PROJ + "/work/render") for f in fs if f.endswith(".png")]
if stills:
    display(Show(sorted(stills)[0], width=900))